# Operational Hygiene for Reasoning
Companion notebook for the **Operational Hygiene for Reasoning** section of Chapter 8.

Clean reasoning means you can inspect a run and understand what happened without guessing. Four practical habits: structured prompts, visible (not hidden-chain-of-thought) logging, cleaning scratch state after each run, and inspectable outputs.

The book's structured-prompt example, as a real (nested) Python object instead of loose YAML text -- keep role/task/constraints/output_format as clearly separate fields:

In [1]:
prompt = {
    "role": "Support resolution agent",
    "task": "Review the case and propose next action",
    "constraints": [
        "Use policy tool before refund decisions",
        "Do not guess account status",
    ],
    "output_format": {
        "action": "<approve|deny|escalate>",
        "reason": "<short text>",
        "evidence": ["<tool result or policy id>"],
    },
}

import json
print(json.dumps(prompt, indent=2))

{
  "role": "Support resolution agent",
  "task": "Review the case and propose next action",
  "constraints": [
    "Use policy tool before refund decisions",
    "Do not guess account status"
  ],
  "output_format": {
    "action": "<approve|deny|escalate>",
    "reason": "<short text>",
    "evidence": [
      "<tool result or policy id>"
    ]
  }
}


**Log the visible steps**, not hidden chain-of-thought: prompt version, model, tool calls and results, retries, latency, final output. A tiny structured log entry for one reasoning step:

In [2]:
run_log = []


def log_step(step_name: str, **fields):
    run_log.append({"step": step_name, **fields})


log_step("prompt_assembly", prompt_version="v3", model="gpt-4o")
log_step("tool_call", tool="policy_lookup", latency_ms=180, retries=0)
log_step("decision", action="approve", evidence=["policy_id: REF-014"])

for entry in run_log:
    print(entry)

{'step': 'prompt_assembly', 'prompt_version': 'v3', 'model': 'gpt-4o'}
{'step': 'tool_call', 'tool': 'policy_lookup', 'latency_ms': 180, 'retries': 0}
{'step': 'decision', 'action': 'approve', 'evidence': ['policy_id: REF-014']}


**Clean state after each run**: keep durable fields explicit, and clear scratch data so it never leaks into the next task.

In [3]:
run_state = {
    "durable": {"case_id": "C-501", "decision": "approve"},
    "scratch": {"draft_notes": "considered escalation, ruled out", "temp_flags": [1, 2]},
}


def end_run(state: dict) -> dict:
    """Only the durable fields survive past the end of a run."""
    return {"durable": state["durable"]}


print("before:", run_state)
print("after: ", end_run(run_state))

before: {'durable': {'case_id': 'C-501', 'decision': 'approve'}, 'scratch': {'draft_notes': 'considered escalation, ruled out', 'temp_flags': [1, 2]}}
after:  {'durable': {'case_id': 'C-501', 'decision': 'approve'}}


Finally, **reduce prompt drift**: version prompts, record why each change was made (`prompt` above already carries a `prompt_version` implicitly through the log), and test major edits before release rather than shipping small unreviewed tweaks.